In [1]:
import duckdb
import pandas as pd
from datetime import timedelta

### About the data:

The Texas Department of Insurance, Division of Workers' Compensation (DWC) maintains a database of medical and pharmacy billing services. It contains charges, payments, and treatments billed by health care professionals and/or medical facilities that treat injured employees, including ambulatory surgical centers. The data is separated into three different types of billing services:

- Professional Medical Billing Services (SV1) - Charges, payments, and treatments billed on a CMS-1500 form by doctors and other health care professionals
- Institutional Medical Billing Services (SV2) - Charges, payments, and treatments billed on a CMS-1450 form (UB-92, UB-04) by hospitals and medical facilities
- Pharmacy Medical Billing Services (SV4) - Charges, payments, and prescriptions billed on a DWC Form-066, Statement of Pharmacy Services

The data for each of these billing services are further seperated into a header (which groups individual line items within a bill) and detail information (individual line items a single bill). you can find this data at data.texas.gov

This data has been transformed into the OMOP data model from OHDSI

In [2]:
# Create connection
con = duckdb.connect('/workspaces/txwc/tx_workers_comp.db')

### What is PDC?

PDC (Proportion of Days Covered) is a key metric used to measure medication adherence. It represents the percentage of days a patient has medication available over a specific time period.

PDC is calculated by dividing the number of days the patient has medication available by the total number of days in the measurement period. The result is expressed as a percentage, with higher percentages indicating better adherence.

For example, if a patient has medication for 270 days out of a 365-day period, their PDC would be:

$$\text{PDC} = \frac{270}{365} \times 100\% = 74\%$$  

PDC is particularly important in:
- Quality measures for healthcare plans (like Medicare Star Ratings)
- Assessing patient compliance with chronic medication regimens
- Identifying patients who may need intervention to improve adherence
- Evaluating pharmacy performance metrics

A PDC of 80% or higher is typically considered good adherence for most chronic medications, though this threshold can vary depending on the specific condition and medication being evaluated.

In [3]:
concept = con.execute('select * from omop.concept').fetchdf()
concept_ancestor = con.execute('select * from omop.concept_ancestor').fetchdf()
drug_exposure = con.execute('select * from omop.drug_exposure').fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

This set of functions takes raw OMOP-format data and prepares a clean list of drug exposures at the ingredient level (ex. 'Metformin ER 500mg' is mapped to 'Metformin' concept). We are also computing a standardized end date for each drug_exposure.

Details:
1. The standardize_omop_types function ensures all key concept IDs in the concept, concept_ancestor, and drug_exposure tables are in a consistent numeric format. This prevents merge errors and ensures clean joins.

2. In build_final_drug_exposures, we pull out only the concepts that are RxNorm Ingredients, then using the concept_ancestor table, we map each drug_concept_id in the exposure table to its ingredient-level concept ID.

3. Then we do some data cleaning - filtering out rows that have unrealistic days_supply ranges and rows with drug_concept_id = 0, meaning the drug_source_concept_id (NDC) value was not mapped to a standard, valid RxNorm concept.

4. Finally, we calculate the standardized end date for each drug exposure based on the start date and days supply. We fill in missing end_date values using start_date + days_supply, and if the days_supply is still missing, we default to a 1 day duration

In [4]:
def standardize_omop_types(concept, concept_ancestor, drug_exposure):
    concept["concept_id"] = pd.to_numeric(concept["concept_id"], errors="coerce").astype("Int64")
    concept_ancestor["ancestor_concept_id"] = pd.to_numeric(concept_ancestor["ancestor_concept_id"], errors="coerce").astype("Int64")
    concept_ancestor["descendant_concept_id"] = pd.to_numeric(concept_ancestor["descendant_concept_id"], errors="coerce").astype("Int64")
    drug_exposure["drug_concept_id"] = pd.to_numeric(drug_exposure["drug_concept_id"], errors="coerce").astype("Int64")
    return concept, concept_ancestor, drug_exposure

def build_final_drug_exposures(concept, concept_ancestor, drug_exposure, max_days_supply=365):
    # Filter RxNorm ingredients
    ingredients = concept[
        (concept["vocabulary_id"] == "RxNorm") &
        (concept["concept_class_id"] == "Ingredient")
    ][["concept_id"]].rename(columns={"concept_id": "ingredient_concept_id"})

    # Join to get ingredient-level exposures
    ingredient_ancestors = concept_ancestor.merge(
        ingredients, left_on="ancestor_concept_id", right_on="ingredient_concept_id"
    )
    exposures = drug_exposure.merge(
        ingredient_ancestors, left_on="drug_concept_id", right_on="descendant_concept_id"
    )

    # Clean and filter exposures
    exposures = exposures[
        (exposures["drug_concept_id"].notna()) &
        (exposures["drug_concept_id"] != 0) &
        (exposures["days_supply"].fillna(0).between(0, max_days_supply))
    ].copy()

    # Parse dates and calculate fallback end dates
    exposures["drug_exposure_start_date"] = pd.to_datetime(exposures["drug_exposure_start_date"], errors="coerce")
    exposures["drug_exposure_end_date"] = pd.to_datetime(exposures["drug_exposure_end_date"], errors="coerce")
    fallback_end = exposures["drug_exposure_start_date"] + pd.to_timedelta(exposures["days_supply"].fillna(0), unit="D")

    exposures["final_drug_exposure_end_date"] = exposures["drug_exposure_end_date"].combine_first(
        fallback_end.where(fallback_end != exposures["drug_exposure_start_date"])
    ).fillna(exposures["drug_exposure_start_date"] + pd.Timedelta(days=1))

    return exposures[[
        "drug_exposure_id", "person_id", "ingredient_concept_id",
        "drug_exposure_start_date", "days_supply", "final_drug_exposure_end_date"
    ]].rename(columns={"final_drug_exposure_end_date": "drug_exposure_end_date"})

# Standardize your OMOP data types
concept, concept_ancestor, drug_exposure = standardize_omop_types(concept, concept_ancestor, drug_exposure)

# Build final exposure table
final_df = build_final_drug_exposures(concept, concept_ancestor, drug_exposure)

final_df.head()


,drug_exposure_id,person_id,ingredient_concept_id,drug_exposure_start_date,days_supply,drug_exposure_end_date
0,1,743539690,1771162,2013-02-11,84,2013-02-11
1,2,367725858,715233,2015-10-01,5,2015-10-06
2,3,138595881,1551099,2015-08-08,5,2015-08-13
3,4,042769557,1177480,2013-11-20,60,2014-01-19
4,5,197991104,924566,2022-03-10,30,2022-04-09


This next step identifies the end dates of continuous drug exposure episodes for each patient and drug ingredient. It does this by analyzing the timeline of exposure start and end dates using a logic that mimics SQL window functions.

Details:
1. Build a timeline of events
Turn each drug fill into two events:
- A start event (when the drug was picked up)
- An end event (when the drug ran out allegedly)

2. Sorts those events chronologically for each patient and ingredient.

3. Assigns ordinal numbers
Basically mimicing SQL's ROW_NUMBER() and MAX(...) OVER (...) to track how many starts and ends we've seen so far for each patient and ingredient

4. Finds balance points
In healthcare data, patients often have multiple drug fills that:
- Overlap (e.g., early refills)
- Are back-to-back (continuous use)
- Or have gaps (indicating possible stops/starts)

To build clean episodes of continuous use, we need a way to figure out where an episode truly ends — that is, when a patient is no longer covered by any of the fills. So we take every fill and split it into a 'start event' (when the medication becomes available) and an 'end event' (when that fill’s supply runs out), then we use the formula `(2 * start_ordinal_filled) - overall_ord == 0` to find where an entire patient's supply of a medication has really run out.

Let me go into more detail about this formula:

`(2 * start_ordinal_filled) - overall_ord == 0`

- start_ordinal_filled = Count of how many exposures have started
- overall_ord = Total events seen (starts + ends)

We multiply starts by 2	because for each episode, we expect 1 start + 1 end. When 2 * starts == total events this means we’ve seen a full pairing of starts and ends — i.e., an episode is complete

I'm basically emulating a SQL window function in pandas.

In [5]:
def generate_episode_end_events(final_df, start_col="drug_exposure_start_date", end_col="drug_exposure_end_date"):
    """
    Takes a final exposure DataFrame and returns end dates of continuous exposure episodes.
    """
    # Create start and end event frames
    start_events = final_df[["person_id", "ingredient_concept_id", start_col]].copy()
    start_events = start_events.rename(columns={start_col: "event_date"})
    start_events["event_type"] = -1

    end_events = final_df[["person_id", "ingredient_concept_id", end_col]].copy()
    end_events = end_events.rename(columns={end_col: "event_date"})
    end_events["event_type"] = 1

    # Combine and sort
    events = pd.concat([start_events, end_events], ignore_index=True)
    events["event_date"] = pd.to_datetime(events["event_date"])
    events.sort_values(by=["person_id", "ingredient_concept_id", "event_date", "event_type"], inplace=True)

    # Assign ordinals
    events["start_ordinal"] = events.groupby(
        ["person_id", "ingredient_concept_id", "event_type"]
    ).cumcount() + 1
    events["start_ordinal"] = events["start_ordinal"].where(events["event_type"] == -1)

    events["overall_ord"] = events.groupby(
        ["person_id", "ingredient_concept_id"]
    ).cumcount() + 1

    events["start_ordinal_filled"] = events.groupby(
        ["person_id", "ingredient_concept_id"]
    )["start_ordinal"].ffill()

    episode_ends = events[
        (2 * events["start_ordinal_filled"] - events["overall_ord"]) == 0
    ][["person_id", "ingredient_concept_id", "event_date"]].rename(columns={"event_date": "end_date"})

    return episode_ends

episode_ends = generate_episode_end_events(final_df)

episode_ends.head()

,person_id,ingredient_concept_id,end_date
19041828,000000877,1000560,2013-07-05
24512610,000000877,1103314,2013-08-06
15582984,000000877,1103314,2013-09-30
26268083,000000877,1103314,2013-11-14
20505939,000000RGOON,1150345,2013-12-26


Next we take the list of drug exposures and episode end dates, and for each exposure, finds the earliest valid end date that occurs on or after the exposure start. It ensures each exposure is assigned to the correct continuous-use episode.

In [6]:
def assign_episode_boundaries(final_df, episode_ends):
    # Join exposures with possible episode ends
    merged = final_df.merge(
        episode_ends,
        on=["person_id", "ingredient_concept_id"],
        how="inner"
    )

    # Keep only end dates that come after the exposure start
    merged = merged[merged["end_date"] >= merged["drug_exposure_start_date"]]

    # Select the earliest valid end date for each exposure
    episode_boundaries = merged.groupby(
        ["drug_exposure_id", "person_id", "ingredient_concept_id", "drug_exposure_start_date"]
    )["end_date"].min().reset_index().rename(columns={"end_date": "drug_sub_exposure_end_date"})

    return episode_boundaries

episode_boundaries = assign_episode_boundaries(final_df, episode_ends)

episode_boundaries.head()

,drug_exposure_id,person_id,ingredient_concept_id,drug_exposure_start_date,drug_sub_exposure_end_date
0,1,743539690,1771162,2013-02-11,2013-02-11
1,2,367725858,715233,2015-10-01,2015-10-06
2,3,138595881,1551099,2015-08-08,2015-08-13
3,4,042769557,1177480,2013-11-20,2014-01-19
4,5,197991104,924566,2022-03-10,2022-07-30


Then we take all the drug exposures that belong to the same patient, drug, and episode end date, and grouped them together. This let us collapse overlapping or adjacent fills into a single episode of continuous use.

For each episode, we:
1. Found the earliest start date

2. Counted how many fills were part of it

3. Gave it a row number to uniquely identify the episode

Now, instead of a list of raw exposures, we have clean, structured episodes — each with a start, end, and count of fills

In [7]:
def summarize_sub_exposure_episodes(episode_boundaries):
    # Group by person, drug, and sub-exposure end to summarize
    grouped = episode_boundaries.groupby(
        ["person_id", "ingredient_concept_id", "drug_sub_exposure_end_date"]
    ).agg(
        drug_sub_exposure_start_date=("drug_exposure_start_date", "min"),
        drug_exposure_count=("drug_exposure_start_date", "count")
    ).reset_index()

    # Assign row numbers within each episode grouping
    grouped["row_number"] = grouped.sort_values(
        by=["person_id", "ingredient_concept_id", "drug_sub_exposure_end_date", "drug_sub_exposure_start_date"]
    ).groupby(
        ["person_id", "ingredient_concept_id", "drug_sub_exposure_end_date"]
    ).cumcount() + 1

    # Reformat columns to match standard output of drug_era table
    final_episodes = grouped[[
        "row_number", "person_id", "ingredient_concept_id",
        "drug_sub_exposure_start_date", "drug_sub_exposure_end_date", "drug_exposure_count"
    ]].rename(columns={"ingredient_concept_id": "drug_concept_id"})

    return final_episodes

final_episodes = summarize_sub_exposure_episodes(episode_boundaries)

final_episodes.head()

,row_number,person_id,drug_concept_id,drug_sub_exposure_start_date,drug_sub_exposure_end_date,drug_exposure_count
0,1,000000877,1000560,2013-07-01,2013-07-05,1
1,1,000000877,1103314,2013-07-27,2013-08-06,1
2,1,000000877,1103314,2013-09-18,2013-09-30,1
3,1,000000877,1103314,2013-11-01,2013-11-14,1
4,1,000000RGOON,1150345,2013-12-11,2013-12-26,1


In this step, we added the duration of each drug 'sub-exposure' episode. We already knew when each episode started and ended, so now we simply calculate how many days the patient was continuously exposed to the medication.

In [8]:
def add_days_exposed(final_episodes):
    episodes = final_episodes.copy()

    # Calculate duration
    episodes["days_exposed"] = (
        pd.to_datetime(episodes["drug_sub_exposure_end_date"]) -
        pd.to_datetime(episodes["drug_sub_exposure_start_date"])
    ).dt.days

    # Rename and reorder columns
    result = episodes.rename(columns={"drug_concept_id": "ingredient_concept_id"})[[
        "row_number", "person_id", "ingredient_concept_id",
        "drug_sub_exposure_start_date", "drug_sub_exposure_end_date",
        "drug_exposure_count", "days_exposed"
    ]]

    return result

final_with_duration = add_days_exposed(final_episodes)

final_with_duration.head()

,row_number,person_id,ingredient_concept_id,drug_sub_exposure_start_date,drug_sub_exposure_end_date,drug_exposure_count,days_exposed
0,1,000000877,1000560,2013-07-01,2013-07-05,1,4
1,1,000000877,1103314,2013-07-27,2013-08-06,1,10
2,1,000000877,1103314,2013-09-18,2013-09-30,1,12
3,1,000000877,1103314,2013-11-01,2013-11-14,1,13
4,1,000000RGOON,1150345,2013-12-11,2013-12-26,1,15


This function adds a grace period of 30 days to each episode’s end and then uses a balance-point method (like we did earlier) to find the true end of a drug era — that is, the last date before a patient went off treatment long enough to break continuity.

In [9]:
def generate_grace_period_end_dates(final_episodes, grace_days=30):
    # Create start and padded end events
    start_events = final_episodes[[
        "person_id", "drug_concept_id", "drug_sub_exposure_start_date"
    ]].rename(columns={"drug_sub_exposure_start_date": "event_date"}).copy()
    start_events["event_type"] = -1
    start_events["start_ordinal"] = start_events.groupby(
        ["person_id", "drug_concept_id"]
    ).cumcount() + 1

    end_events = final_episodes[[
        "person_id", "drug_concept_id", "drug_sub_exposure_end_date"
    ]].copy()
    end_events["event_date"] = end_events["drug_sub_exposure_end_date"] + pd.Timedelta(days=grace_days)
    end_events["event_type"] = 1
    end_events["start_ordinal"] = None

    # Combine and sort events
    event_df = pd.concat([start_events, end_events], ignore_index=True)
    event_df.sort_values(by=["person_id", "drug_concept_id", "event_date", "event_type"], inplace=True)

    # Forward fill start ordinal and assign overall order
    event_df["start_ordinal_filled"] = event_df.groupby(
        ["person_id", "drug_concept_id"]
    )["start_ordinal"].ffill()
    event_df["overall_ord"] = event_df.groupby(
        ["person_id", "drug_concept_id"]
    ).cumcount() + 1

    # Apply episode end logic
    grace_end_dates = event_df[
        (2 * event_df["start_ordinal_filled"] - event_df["overall_ord"]) == 0
    ][["person_id", "drug_concept_id", "event_date"]].copy()

    # Adjust back by grace period
    grace_end_dates["end_date"] = grace_end_dates["event_date"] - pd.Timedelta(days=grace_days)

    # Rename for consistency
    return grace_end_dates.rename(columns={"drug_concept_id": "ingredient_concept_id"})[
        ["person_id", "ingredient_concept_id", "end_date"]
    ]

grace_end_dates = generate_grace_period_end_dates(final_episodes, grace_days=30)

grace_end_dates.head()

,person_id,ingredient_concept_id,end_date
9408167,000000877,1000560,2013-07-05
9408168,000000877,1103314,2013-08-06
9408169,000000877,1103314,2013-09-30
9408170,000000877,1103314,2013-11-14
9408171,000000RGOON,1150345,2013-12-26


We took our list of sub-exposure episodes (each with a start and end), and we linked each one to the end of its broader “drug era.” To do that, we used the grace period episode ends from the last step. For each episode start, we found the earliest valid grace period end that came after it. 

This lets us group episodes that are close together (within 30 days) into a single drug era — which is a more realistic view of how the patient was continuously taking the drug.

In [10]:
def assign_drug_era_end_dates(final_episodes, grace_end_dates):
    episodes = final_episodes.copy()

    # Ensure days_exposed is present
    if "days_exposed" not in episodes.columns:
        episodes["days_exposed"] = (
            pd.to_datetime(episodes["drug_sub_exposure_end_date"]) -
            pd.to_datetime(episodes["drug_sub_exposure_start_date"])
        ).dt.days

    # Merge with grace period end dates
    merged = episodes.merge(
        grace_end_dates,
        left_on=["person_id", "drug_concept_id"],
        right_on=["person_id", "ingredient_concept_id"],
        how="inner"
    )

    # Keep only end dates that follow or match the start of the episode
    merged = merged[merged["end_date"] >= merged["drug_sub_exposure_start_date"]]

    # Find earliest valid end date (drug era end) per episode
    drug_eras = merged.groupby([
        "person_id", "drug_concept_id", "drug_sub_exposure_start_date",
        "drug_exposure_count", "days_exposed"
    ])["end_date"].min().reset_index().rename(columns={"end_date": "drug_era_end_date"})

    return drug_eras

drug_eras = assign_drug_era_end_dates(final_episodes, grace_end_dates)
drug_eras.head()

,person_id,drug_concept_id,drug_sub_exposure_start_date,drug_exposure_count,days_exposed,drug_era_end_date
0,000000877,1000560,2013-07-01,1,4,2013-07-05
1,000000877,1103314,2013-07-27,1,10,2013-08-06
2,000000877,1103314,2013-09-18,1,12,2013-09-30
3,000000877,1103314,2013-11-01,1,13,2013-11-14
4,000000RGOON,1150345,2013-12-11,1,15,2013-12-26


This step groups drug sub-exposure episodes into longer drug eras, calculates their start and end, counts how many fills occurred, and computes how many gap days (days uncovered) exist within the era.

In [11]:
def summarize_drug_eras(drug_eras):
    # Group by patient, drug, and era end to summarize
    summary = drug_eras.groupby(
        ["person_id", "drug_concept_id", "drug_era_end_date"]
    ).agg(
        drug_era_start_date=("drug_sub_exposure_start_date", "min"),
        drug_exposure_count=("drug_exposure_count", "sum"),
        total_days_exposed=("days_exposed", "sum")
    ).reset_index()

    # Calculate gap days
    summary["gap_days"] = (
        (summary["drug_era_end_date"] - summary["drug_era_start_date"]).dt.days -
        summary["total_days_exposed"]
    )

    # Assign unique era ID per patient
    summary["drug_era_id"] = summary.sort_values("person_id").groupby("person_id").cumcount() + 1

    # Select and order final columns
    return summary[[
        "drug_era_id", "person_id", "drug_concept_id",
        "drug_era_start_date", "drug_era_end_date",
        "drug_exposure_count", "gap_days"
    ]]

drug_era_summary = summarize_drug_eras(drug_eras)
drug_era_summary.head()

,drug_era_id,person_id,drug_concept_id,drug_era_start_date,drug_era_end_date,drug_exposure_count,gap_days
0,1,000000877,1000560,2013-07-01,2013-07-05,1,0
1,2,000000877,1103314,2013-07-27,2013-08-06,1,0
2,3,000000877,1103314,2013-09-18,2013-09-30,1,0
3,4,000000877,1103314,2013-11-01,2013-11-14,1,0
4,1,000000RGOON,1150345,2013-12-11,2013-12-26,1,0


This step takes the finalized drug eras and calculates PDC by dividing the days covered (i.e. non-gap days) by the total number of days in the era

In [12]:
def calculate_pdc(drug_era_summary):
    summary = drug_era_summary.copy()

    # Calculate era length
    summary["era_length_days"] = (
        summary["drug_era_end_date"] - summary["drug_era_start_date"]
    ).dt.days + 1

    # Calculate PDC
    summary["pdc"] = (
        (summary["era_length_days"] - summary["gap_days"]) /
        summary["era_length_days"]
    ) * 100

    # Round to 2 decimal places
    summary["pdc"] = summary["pdc"].round(2)

    return summary

drug_era_with_pdc = calculate_pdc(drug_era_summary)
drug_era_with_pdc.head()

,drug_era_id,person_id,drug_concept_id,drug_era_start_date,drug_era_end_date,drug_exposure_count,gap_days,era_length_days,pdc
0,1,000000877,1000560,2013-07-01,2013-07-05,1,0,5,100.0
1,2,000000877,1103314,2013-07-27,2013-08-06,1,0,11,100.0
2,3,000000877,1103314,2013-09-18,2013-09-30,1,0,13,100.0
3,4,000000877,1103314,2013-11-01,2013-11-14,1,0,14,100.0
4,1,000000RGOON,1150345,2013-12-11,2013-12-26,1,0,16,100.0


In [13]:
# Get rows where PDC is less than 80
pdc_below_80 = drug_era_with_pdc[
    (drug_era_with_pdc["pdc"] < 80) &
    (drug_era_with_pdc["pdc"] >= 0)
].copy()

pdc_below_80.head()

,drug_era_id,person_id,drug_concept_id,drug_era_start_date,drug_era_end_date,drug_exposure_count,gap_days,era_length_days,pdc
28,1,000004236,704943,2012-10-02,2012-12-05,6,22,65,66.15
30,3,000004236,1115008,2012-08-24,2012-12-05,11,32,104,69.23
47,4,000008491,1150345,2016-07-22,2016-10-28,3,29,99,70.71
90,26,000013049,1124957,2024-06-05,2024-09-10,7,44,98,55.10
112,5,000013283,1506270,2021-05-03,2021-05-17,3,8,15,46.67


Most of the code (besides the pdc calculation) is used to create the drug_era omop table. I just wanted to go into a little detail about how pharmacy data needs to be cleaned and transformed before we can calculate PDC.

In [14]:
con.close()